## 8. Reinforcement Fine-Tuning (RFT)

Supervised fine-tuning (Section 7) significantly improved the Bank
Agent on short, policy-driven tool sequences such as:

- name/ZIP lookup → profile fetch  
- simple order inquiries  
- address-change flows  

However, some customer scenarios require **much deeper multi-step reasoning**
across multiple tool calls. These include:

- reviewing an entire order history  
- checking return eligibility for multiple items  
- computing refund paths (gift card vs payment method)  
- verifying policy exceptions  
- branching logic depending on item category, time window, or status  

These complex workflows often span **7–10 tool calls** and require the
model to *reason* about policy, not merely imitate the next step.

This is where **Reinforcement Fine-Tuning (RFT)** becomes essential.

![RFT 101](./img/rft_101.png)


### 8.1 What RFT Optimizes in the Bank Agent

RFT trains the model to maximize a **reward** computed over the *entire*
multi-turn output rather than predicting the next tool call in isolation.

In our case, the reward comes from a **custom LLM-based grader** that checks:

1. **Return-policy correctness**  
   - Did the model apply the correct time windows?
   - Did it respect category-specific restrictions?
   - Did it choose the right remedy (refund, exchange, store credit)?

2. **Sequence correctness**  
   - Did the model choose the needed tools in the correct order?
   - Did it avoid redundant or invalid tool calls?

3. **Argument correctness**  
   - Are the tool-call inputs valid and consistent with prior tool outputs?

4. **Outcome correctness**  
   - Is the final summary aligned with the expected customer guidance?

SFT teaches *how* to perform tool chaining.  
RFT teaches *why* a sequence is right or wrong — and how to optimize it.

This is especially powerful when the workflow has branching logic like:

- “If the item is in category X and within 15 days, allow return;  
  otherwise offer store credit or deny.”

Even a strong base model struggles with such decision-heavy scenarios unless
it is trained with **reward-driven optimization** over long conversations.

In the next sections, we prepare the RFT dataset, design the reward function,
and run an RFT job that dramatically improves the agent's ability to follow
deep return-policy logic.


### 8.2 The RFT Dataset (Multi-Order, Policy-Heavy Scenarios)

Reinforcement Fine-Tuning requires a different dataset than SFT.  
Instead of teaching the model to **imitate** a transcript, RFT teaches the model to **optimize correctness** across complex, multi-item return-policy scenarios using a reward function.

The dataset used in this notebook lives in:

```bash
data/rft/rft_train.jsonl
data/rft/rft_test.jsonl
```

Each record includes:

- A multi-turn conversation (often 7–12 turns)
- Multiple orders and multiple item_ids in the same request
- Mixed eligibility windows (electronics vs general merchandise vs defect-only)
- Account-level restrictions (e.g., store-credit-only refunds)
- Prior tool outputs that the model must interpret
- Several items with **different policies and outcomes** in a single turn

These scenarios mimic real customer-service complexity and force the model to reason about:

- purchase dates  
- category rules  
- return windows  
- defect conditions  
- refund routing rules  
- multi-step logical dependencies  

This type of reasoning **cannot** be learned reliably through SFT alone.

#### Why This Dataset Is a Good Fit for RFT

The dataset is deliberately “policy-heavy” and contains scenarios such as:

- 7–10 tool calls across a single workflow  
- Mixed eligibility in the same conversation  
- Conflicting policy dimensions (category × days × condition × account rules)  
- Cross-order dependencies  
- Conversations where correctness can’t be judged by matching a transcript  

SFT teaches the syntax of calling tools.  
RFT teaches **decision-making quality**.

Next, we design the **reward function** (Section 8.3), which evaluates an entire tool-calling workflow and returns a numeric score.  
This becomes the signal the model uses during RFT training.


### 8.3 Designing the Reward Function (LLM‑Based Return‑Policy Grader)

Reinforcement Fine‑Tuning (RFT) needs a **numeric reward** for each conversation.
In this cookbook, that reward comes from a model‑based grader that judges how well
the assistant applied Zava’s return policy for every item the customer asks about.

The grader configuration is stored in:

```bash
data/rft/rft_grader-config.json
```

At a high level, the reward model receives three things:

- The **customer request and conversation history**
- The **structured reference outcome** for each item (what should happen)
- The **assistant’s final explanation**, including its per‑item decisions

It then produces a score between **0.0 and 1.0**, where higher is better.

#### 8.3.1 What the Grader Checks

For each item in the request, the grader evaluates:

1. **ID Matching**
   - Does the assistant mention the correct `order_id` and `item_id`?
   - If IDs are wrong or missing, the score for that item is driven toward 0.0.

2. **Eligibility Decision**
   - Did the assistant correctly decide whether the item is return‑eligible,
     exchange‑eligible, or not eligible at all?
   - Incorrect eligibility (e.g., saying “eligible” when it is not) yields a
     zero score for that item.

3. **Policy Justification**
   - Does the explanation reference a concrete policy detail?
     Examples include:
       - time window (15‑day electronics, 30‑day general goods)
       - item condition (opened vs unopened)
       - account flags (store‑credit‑only)
       - marketplace or third‑party seller rules
   - If the eligibility is correct but the explanation is vague, the item gets
     partial credit (e.g., 0.5 instead of 1.0).

4. **Overall Consistency**
   - The explanation must not contradict itself across items.
   - The final summary must align with the per‑item outcomes.

The final reward for a conversation is the **average** of all per‑item scores,
normalized into the range **[0.0, 1.0]**.

#### 8.3.2 Why This Reward Works Well for RFT

This reward function is tightly aligned with what a retail business actually
cares about:

- Every item is judged **individually**, so the model can’t “hide” mistakes.
- The model is pushed to **explain the policy**, not just guess “yes/no”.
- Hallucinations and ID mismatches are penalized heavily.
- Correct but unjustified answers are only partially rewarded.

During RFT, the model repeatedly generates candidate responses for these
scenarios, receives a reward from this grader, and updates its parameters to
maximize that reward. Over time, it learns to produce:

- policy‑consistent decisions,
- grounded references to the right orders and items,
- and clear, customer‑friendly explanations.

In the next section (8.4), we’ll see how to launch an RFT job that uses this
reward function together with the `rft_train.jsonl` dataset.


### 8.4 Launching the RFT Job (UI + Optional Code)

Reinforcement Fine-Tuning (RFT) uses three components:

1. **Training dataset**
2. **Reward function (grader)**
3. **Base model**

All three are already prepared in this cookbook.

#### 8.4.1 Files Used for RFT

Training set:

```bash
data/rft/rft_train.jsonl
```

Evaluation (holdout) set:

```bash
data/rft/rft_test.jsonl
```

Reward function configuration:

```bash
data/rft/rft_grader-config.json
```

Custom tools metadata (used by the reward model):

```bash
data/rft/rft_tools-config.json
```

### 8.4.2 Launching an RFT Job

In [ ]:
# Upload train file
with Path("data/rft_train_fixed.jsonl").open("rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
print(f"Train File ID: {train_file.id}")

# Upload validation file
with Path("data/rft_test_fixed.jsonl").open("rb") as f:
    valid_file = client.files.create(file=f, purpose="fine-tune")
print(f"Valid File ID: {valid_file.id}")

In [ ]:

# Configure reinforcement learning parameters
with open("data/rft_grader-config.json", 'r') as file:
    grader_config = json.load(file)

with open("data/rft_tools-config.json", 'r') as file:
    tool_server_url = os.getenv("TOOLS_SERVER_URL", "#TOOLS_SERVER_URL#")
    rft_tool_config = file.read().replace('#TOOLS_SERVER_URL#', tool_server_url)
    tools_config = json.loads(rft_tool_config)

method_body = {
    "type": "reinforcement",
    "reinforcement": {
      "hyperparameters": {
        "eval_interval": 3,
        "eval_samples": 5,
        "compute_multiplier": 1,
        "reasoning_effort": "medium",
        "n_epochs": 6,
        "batch_size": 2,
        "learning_rate_multiplier": 2
      },
      "grader": grader_config,
      "tools": tools_config
    }
  }

print("Method Body:")
print(json.dumps(method_body, indent=2))

In [ ]:
# Create fine-tuning job
response = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=valid_file.id,
    model = "o4-mini-2025-04-16",
    suffix="bank-sft-1",
    method= method_body,    
    extra_body={"trainingType": "GlobalStandard"}
)

print(f"Job ID: {response.id}")
print(f"Status: {response.status}")

### 8.5 Monitoring and Interpreting RFT Training Metrics

Reinforcement Fine-Tuning produces more nuanced metrics than SFT.  
Instead of simply tracking loss curves, RFT logs **reward trends** over time.
These reward curves are essential for validating that the model is learning
policy-aware decision-making.

### 8.5.1 Key Metrics to Watch

When you open the RFT job details in Microsoft Foundry Studio, look for:

#### **1. Average Reward per Step**
- The most important metric.
- Should trend upward over the course of training.
- Early plateaus are normal; late-stage jumps indicate the model discovered
  a better policy pattern.

#### **2. Reward Variance**
- Measures consistency.
- High variance early on means the model is exploring multiple strategies.
- Over time, variance should narrow as the model locks onto stable policy reasoning.

#### **3. Rejection / Invalid Output Rate**
- Indicates how often the reward model could not parse the assistant’s output.
- Should decrease sharply after the first few hundred steps.
- If this stays flat, the reward function may need adjustment.

### 8.5.2 Viewing Metrics in the UI

In the Microsoft Foundry interface, navigate to:

**Model Customization → Reinforcement Fine-Tuning → [Your RFT Job] → Metrics**

You will see:

- Reward progression chart  
- Per-item scoring distribution  
- Training event logs  
- Model-checkpoint-by-checkpoint comparisons  

These views help you validate that your reward function and dataset are producing stable learning.
![RFT Metrics](./img/rft_metrics.png)

The screen shot below show improvement in evaluation result at every step of training.
![RFT Auto Evals](./img/rft_auto_evals.png)

### 8.5.3 How to Interpret Training Curves

A **healthy RFT run** typically shows:

- A steady climb in average reward  
- A reduction in invalid outputs  
- A decrease in reward variance  
- Occasional jumps when the model discovers a better decision pattern  

Unhealthy signs include:

- Flat reward curves  
- Rising invalid output rates  
- Collapsing to trivial strategies (e.g., always rejecting returns)  
- Overfitting (training reward rises but eval reward drops)

![RFT Rewards](./img/rft_rewards.png)

### 8.5.4 The Goal of Monitoring

Monitoring ensures that:

- The model is learning **policy reasoning**, not superficial correlations.  
- Tool arguments become more accurate over time.  
- The assistant becomes more consistent across multi-item workflows.  
- Reward-model alignment remains tight throughout training.

Once you see stable improvements across training and evaluation sets,
you are ready to test the RFT model in live agent flows (Section 8.6).

![RFT Sample Scores](./img/rft_sample_scores.png)

![RFT Sample Scores Progression](./img/rft_sample_scores_progression.png)

![RFT Sample Output Comparision](./img/rft_sample_output_comparision.png)


### 8.6 Testing the RFT Model in the Bank Agent

After RFT training completes, the final and most satisfying step is to
**test the RFT-enhanced model inside the real Retail Agent workflow**.
This is where we verify that the model has internalized policy reasoning,
long tool chains, and multi-item logic — not because it memorized examples,
but because it learned from reward signals.

### 8.6.1 Switching the Agent to the RFT Model

In your terminal-based demo client (`tools/retail_agent.py`), simply pass the
new RFT model name. It will look similar to:

```bash
python tools/retail_agent.py --model <your-rft-model-name>
```

You can also test the RFT model directly inside Microsoft Foundry:

**AI Foundry → Agents → [Bank Agent] → Test**

Insert a GIF or screenshot of this step: **{TBD}**

### 8.6.2 What You Should Expect to See

With a strong reward function and high-quality RFT dataset, you will notice:

#### **1. Longer, Correct Tool Chains**
The model correctly:

- fetches all orders  
- finds item-level metadata  
- checks eligibility windows  
- interprets category-specific rules  
- follows account exceptions  
- computes refund method correctly  

Often **7–10 API calls** will be invoked in a single user flow.

#### **2. Reduced Hallucinations**
The RFT model is far less likely to:

- invent new tool names  
- fabricate item_ids  
- skip required steps  
- contradict the return policy  

#### **3. Better Explanations**
Beyond correctness, responses are more:

- structured  
- grounded in policy  
- consistent across items  
- helpful to the end user  

You will see explanations referencing the right dates, categories, and
eligibility windows without prompting.

### 8.6.3 Example Terminal Test Flow

Add a GIF showing your real terminal run: **{TBD}**

```bash
# Placeholder for your animation:
# ![RFT Terminal Session](img/rft_terminal_demo.gif)
```

This should show:

- User requesting returns for multiple items  
- The agent invoking a long sequence of tools  
- The final output aligning with the reward model’s expectations  

---

### 8.6.4 Validating the RFT Model

Replaying previous examples with finetuned models, you will observe:

- How the RFT model corrects the exact failures seen in the base and SFT models  
- How multi-item scenarios now behave consistently  
- How policy-heavy cases are handled without hand-designed prompting  

The RFT model is now ready for production-style evaluations or integration
into more complex agent architectures.

Next, we wrap up with Section 9 — a brief conclusion and recommendations.

---

## 9. Summary and Next Steps

This cookbook walked through the complete journey of improving tool‑calling
accuracy for the Bank Agent — from diagnosing model failures, to
synthetic data generation, to SFT, and finally RFT with a policy‑aware reward
function.

By combining these techniques, we demonstrated a practical, production‑ready
approach to building robust enterprise agents in Microsoft Foundry.

### 9.1 What We Accomplished

#### **1. Diagnosed a real failure mode**
- Base models struggled with user‑identity resolution and tool chaining.
- Even strong models made mistakes in multi‑step reasoning (find_user → get_user_details).
- More complex, policy‑heavy workflows needed something beyond SFT.

#### **2. Set up a complete reproducible environment**
- MCP server exposing 17 tools  **{TBD}**
- Local agent client for fast iteration  
- Microsoft Foundry for model customization  
- Clean, versioned datasets in `data/`

#### **3. Generated high‑quality synthetic data**
- Used OpenAPI‑driven generation in Microsoft Foundry  
- Produced thousands of conversations with realistic tool behavior  
- Built SFT-ready train/test splits  
- Added a visualization dashboard and filtering pipeline

#### **4. Built a Python‑based evaluation framework**
- Expanded evaluation sets to one‑record‑per‑tool‑call  
- Wrote a deterministic grader for tool accuracy  
- Benchmarked multiple base models and measured real failures

#### **5. Improved accuracy with Supervised Fine-Tuning**
- Taught the model correct tool chaining  
- Reduced hallucinations  
- Achieved measurable accuracy improvements across all models  
- Verified correctness using the evaluation pipeline

#### **6. Tackled complex policy reasoning with RFT**
- Created long, multi-item, multi-order scenarios  
- Designed an LLM-based policy grader  
- Used reward signals instead of imitation learning  
- Obtained major gains in reasoning correctness and consistency

#### **7. Validated improvements in real flows**
- Ran both SFT and RFT models through the real Retail Agent  
- Observed fewer errors, better tool sequences, and more grounded explanations  
- Terminal demos confirmed correctness in end-to-end workflows


### 9.2 When to Use SFT vs. RFT

| Technique | Best For | Not Ideal For |
|----------|----------|----------------|
| **SFT** | Deterministic tool chaining, argument propagation, syntax learning | Deep reasoning, multi-step decision workflows |
| **RFT** | Policy enforcement, long tool chains, multi-item logic, dynamic decision making | Pure imitation tasks or small-data scenarios |

A production agent often benefits from **both**:
- SFT for predictable behavior  
- RFT for complex reasoning  


### 9.3 Where to Go from Here

#### **1. Extend your reward function**
- Add tracking for latency, brevity, user satisfaction, or policy strictness  
- Combine multiple reward types (e.g., correctness + helpfulness)

#### **2. Build richer tools**
- Add new tool families (inventory, promotions, subscription management)  
- Use tool metadata to guide reward shaping

#### **3. Evaluate on real user logs**
- Replace synthetic datasets with anonymized real customer flows  
- Use error patterns to design new RFT tasks

#### **4. Deploy & observe**
- Integrate your custom models into live agents  
- Use Microsoft Foundry’s monitoring to track drift and failures  

### 9.4 Final Thoughts

This cookbook demonstrates a reproducible, end-to-end pattern for building
high-quality enterprise agents:

- Synthetic data for coverage  
- SFT for structure  
- RFT for deep reasoning  
- Evals for measurement  
- MCP for grounded execution

This pattern generalizes far beyond retail: healthcare, finance, support,
field operations, travel, logistics, and more.

You now have everything you need to design, train, evaluate, and ship
domain‑specialized agent models with confidence.

Happy building! 🚀



![Title Diagram](./img/outro.png)

## 10. Cookbook Appendix (Reusable Recipes)
- How to design datasets
- How to implement graders
- How to generate synthetic data
- How to debug tool-calling traces
- How to choose SFT vs RFT vs DPO
- Cost estimation tools